# Index spread moves

Pick an index (`HY47`, `HY42`, `XO44`, `IG45`, `MA40`, ...) and two dates. First it shows which names are missing data, then one row per index member with the spread, `upfront`, `all_upfront` and `risky_pv01` at both dates and the change in each, sorted by whichever change you pick (`SORT_BY`).

- Reads the local store `cache/spreads.parquet` (no Databricks query). To add newer dates, run `cs.update_store()` (needs a running Bloomberg Terminal and a Databricks login) in the refresh cell below.
- Spread is `conv_spread` in bp at the index maturity; `upfront` and `all_upfront` are in points. Dates snap to the last available date on or before the request, and an index has no data before its launch date.
- After `cs.update_store()`, refresh the derived files with the system Python (they need the Quantifi Toolkit / Excel, which the notebook kernel doesn't have): `python src/cds_pricer.py` (re-prices every priced point at its index coupon into `cache/priced_at_index_coupon.parquet`), and `python src/addin_fill.py pull` then `python src/addin_fill.py assemble` (fills the cells Databricks cannot price from the Markit add-in into `cache/addin_fill.parquet`). Without those files, spreads are shown at the coupon Markit quoted and unpriced cells stay blank.
- Names with no price on a date stay in the table with blank values and a `status` saying why: `missing` (no Databricks row, red). Rows shaded amber are wide names (`recal`), blue rows are re-priced from the other coupon (`recoupon`), purple rows come from the Markit add-in (`addin`). A `spread_override` from `overrides.csv` beats all of these.

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'cdx_spreads.py').exists())     # project root, wherever the notebook is opened from
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import cdx_spreads as cs
from IPython.display import HTML, display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

# no-wrap HTML table with fixed decimals, blanks for missing, red rows for names missing data
show = cs.show_table


print(', '.join(cs.load_index_list()['index_name']))

## Refresh the store (optional)

In [ ]:
# cs.update_store()          # all 40 indices, only fetches dates the store doesn't have yet
# cs.update_store(['HY47'])  # or just some indices

## Parameters

In [ ]:
INDEX = 'HY47'
DATE_X = '2026-09-29'   # start
DATE_Y = '2026-10-23'   # end
SORT_BY = 'spread'      # which change to sort on: 'spread', 'upfront', 'all_upfront' or 'risky_pv01'
SORT_BY_ABS = False     # False: largest positive change on top (widest first for spreads). True: largest move in either direction first.

## 1. Names not priced
**Priced** means a curve point has a `par_spread`, or a `conv_spread` / `upfront` together with the coupon it is quoted at (`risky_pv01` is optional: `cds_pricer.py` derives it from a flat curve). A `conv_spread` of 0 is Markit's 'no spread solves' placeholder, not a price. The goal is for every name on every date to be priced.

Sources, in order: **Databricks** first, then the **Markit add-in** (`addin_fill.py`) for cells Databricks could not price, then any manual override. Every spread, upfront, `all_upfront` and `risky_pv01` below is **at the index coupon** (500 for HY/XO, 100 for IG/MA).

- **Red box:** names with no price on one of the dates, grouped by reason (*never priced in this index*, *conv=0*, *incomplete* stub row, *gap*).
- **Amber box:** wide names priced with a marked-down recovery (`recovery = 100 - all_upfront - 5`, flat 4% discounting): a clean comparable spread, but not Markit's.
- **Blue box:** names Markit quotes at the other coupon, re-priced to the index coupon with `cds_pricer.py`.
- **Purple box:** names priced from the Markit add-in because Databricks has no price (renamed or missing curves); half-year tenors are interpolated between annual nodes.

The table gives, for each flagged name, `days_with_data` of `days_in_index` (priced by either source), `first_data`, the coupon it was quoted at and the recovery used.

In [ ]:
res = cs.spread_changes(INDEX, DATE_X, DATE_Y)
a = res.attrs
own, recoupon, recal, addin, unpriced = cs.priced_counts(res)
print(f"{INDEX}: conv_spread {a['date_x']:%Y-%m-%d} -> {a['date_y']:%Y-%m-%d} | index maturity {a['maturity']:%Y-%m-%d} | index coupon {a['index_coupon']} | "
      f"{len(res)} names: {own} priced by Databricks at the index coupon, {recoupon} re-priced from the other coupon, {recal} wide names at a marked-down recovery, "
      f"{addin} priced from the Markit add-in, {unpriced} not priced")
display(HTML(cs.missing_banner_html(res)))
show(cs.missing_report(res), numeric={'quoted coupon X': 0, 'quoted coupon Y': 0, 'recovery X (%)': 1, 'recovery Y (%)': 1})

## 2. Largest moves
Sorted by the change chosen in `SORT_BY` (see `SORT_BY_ABS` above). Row color shows where a name's numbers came from: plain rows are white on black, tinted rows are black on a light tint.

| Row shading | Meaning |
|---|---|
| black (no tint) | priced by Databricks at the index coupon |
| blue | Markit quotes the name at the other coupon; re-priced to the index coupon with `cds_pricer` |
| purple | Databricks has no price on at least one date; taken from the Markit add-in |
| amber, bold | wide name: no conventional spread solves at 40% recovery, so it is priced at a marked-down recovery (recovery columns show which) |
| red, bold | no price on at least one of the two dates; blank spreads, sorted to the bottom |

In [ ]:
print(f"median spread change {res['change'].median():.1f} bp | median pct change {res['pct_change'].median():.1f}%")

sort_col = {'spread': 'change', 'upfront': 'upfront_change', 'all_upfront': 'all_upfront_change', 'risky_pv01': 'risky_pv01_change'}[SORT_BY]
res = res.sort_values(sort_col, key=abs if SORT_BY_ABS else None, ascending=False, na_position='last').reset_index(drop=True)

dx, dy = f"{a['date_x']:%Y-%m-%d}", f"{a['date_y']:%Y-%m-%d}"
cols = {'conv_spread_x': f'spread {dx}', 'conv_spread_y': f'spread {dy}', 'change': 'spread change (bp)', 'pct_change': 'spread change (%)'}
for m in ('upfront', 'all_upfront', 'risky_pv01'):
    cols |= {f'{m}_x': f'{m} {dx}', f'{m}_y': f'{m} {dy}', f'{m}_change': f'{m} change'}
cols |= {'recovery_x': f'recovery used {dx} (%)', 'recovery_y': f'recovery used {dy} (%)'}   # only filled on recalibrated rows
view = res[['member_name', 'Market_CDS_Ticker', *cols, 'status']].rename(columns=cols)
places = {name: (2 if name.startswith('risky_pv01') else 1) for name in cols.values()}   # risky_pv01: 2 decimals, everything else 1
show(view, numeric=places, flag_col='status')   # red = missing data, amber = recalibrated

## 3. Pricing coverage, all indices
How much of the store is priced, per index (cells = index x member x date). `priced` is Databricks by the agreed definition; `quoted at index coupon %` counts cells Markit quotes at the index's own coupon (the rest are re-priced). `filled from add-in` is how many of the `not priced` cells the Markit add-in prices, `priced % incl. add-in` the resulting coverage, and `still not priced` what is left. `conv=0 (priced via par)` counts wide-name rows that Markit gives no conv_spread but which still count as priced.

In [ ]:
show(cs.pricing_coverage(), numeric={'priced %': 1, 'quoted at index coupon %': 1, 'priced % incl. add-in': 1})

## 4. Index metrics
Built from each name's `all_upfront` (its present value of expected loss, in points; Markit's value from Databricks where priced there, else the Markit add-in). One row per index and date.

- **dispersion**: standard deviation of `all_upfront` across the index's priced names (unweighted, population std).
- **index_el**: index expected loss = sum of weight x `all_upfront` over the priced names, divided by the weight of all names live on that date (points).
- **top5_share / top10_share**: expected loss of the 5 / 10 widest names (largest `all_upfront`) x their weights, divided by the total over priced names.

Names that defaulted out of a series (from `teams.structured_credit_abhutra.credit_events`) stay in it at their original weight until `index_ex_date`, using their own prices (Databricks, else the Markit add-in), so earlier dates still include them. Unpriced names are left out of that date; `n_priced` and `priced_weight` show how much of the live index each number covers.

In [ ]:
import index_metrics as im

met = im.index_metrics(INDEX).set_index('published_date')
two = met.loc[[a['date_x'], a['date_y']], ['dispersion', 'index_el', 'top5_share', 'top10_share', 'max_el', 'n_priced', 'n_members', 'priced_weight']].copy()
two.loc['change'] = two.iloc[1] - two.iloc[0]
for c in ('top5_share', 'top10_share', 'priced_weight'):
    two[c] = two[c] * 100                       # show as %
two = two.rename(columns={'top5_share': 'top5 share (%)', 'top10_share': 'top10 share (%)', 'priced_weight': 'priced weight (%)', 'index_el': 'index EL (pts)', 'max_el': 'widest (pts)'})
two.insert(0, 'date', [f"{a['date_x']:%Y-%m-%d}", f"{a['date_y']:%Y-%m-%d}", 'change'])
show(two, numeric={'dispersion': 2, 'index EL (pts)': 2, 'top5 share (%)': 1, 'top10 share (%)': 1, 'widest (pts)': 1, 'n_priced': 0, 'n_members': 0, 'priced weight (%)': 1})

The widest names on the end date, and what share of the index's expected loss each carries:

In [ ]:
w = im.widest(INDEX, a['date_y'], 10)
w['weight'] = w['weight'] * 100
w['share_of_index_el'] = w['share_of_index_el'] * 100
w = w.rename(columns={'all_upfront': 'all_upfront (pts)', 'weight': 'weight (%)', 'el_contribution': 'EL contribution (pts)', 'share_of_index_el': 'share of index EL (%)'})
show(w, numeric={'all_upfront (pts)': 1, 'weight (%)': 2, 'EL contribution (pts)': 2, 'share of index EL (%)': 1})